In [1]:
# ===== Colab Setup =====

REPO_URL = "https://github.com/dariuscparker/Self-Improving-Agent-Projects.git"
REPO_ROOT = "/content/Self-Improving-Agent-Projects"
WORKDIR = f"{REPO_ROOT}/modules/01_test_time_compute"

import os
import sys
import subprocess

# Remove any old copy (optional)
if os.path.exists(REPO_ROOT):
    subprocess.run(["rm", "-rf", REPO_ROOT], check=True)

# Clone fresh repo
subprocess.run(["git", "clone", REPO_URL, REPO_ROOT], check=True)

# Verify expected directories exist
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f"Repo root not found: {REPO_ROOT}")

if not os.path.isdir(WORKDIR):
    raise FileNotFoundError(f"Expected notebook directory not found: {WORKDIR}")

# Add repo root to Python path
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Move into working directory
os.chdir(WORKDIR)

# Verify
print("Current working directory:", os.getcwd())
print("Top-level files here:", os.listdir(".")[:20])

Current working directory: /content/Self-Improving-Agent-Projects/modules/01_test_time_compute
Top-level files here: ['hw1-test-time-compute.ipynb']


# HW: Test-Time Compute Strategies on GSM8K

This notebook reflects the exercises generated by ChatGPT to review the course material in module one of Stanford's CS239A course. In this notebook, we will explore various test time compute strategies to help improve the results of a small model.

The first module illustrates that 'more compute' is typically associated with better performance. Using strategies to increase prompting can help our end result.

We will use the **GSM8K** dataset and a **lightweight open model** (**Qwen/Qwen2.5-1.5B-Instruct**) with 4-bit quantization. We will show how a lightweight model can achieve improved results with test time compute strategies. We will explore the following techniques in this notebook:

- **Part 1:** Single-pass CoT baseline (System-1)
- **Part 2:** Best-of-N sampling + **majority vote** (self-consistency)
- **Part 3:** Best-of-N + **simple verifier** (Proposer–Verifier using labels)
- **Part 4:** **Difficulty binning** (Easy/Medium/Hard) via empirical pass rate
- **Part 5:** **Sequential revisions** (self-refinement chains)
- **Part 6:** Lightweight **beam/search** over partial reasoning
- **Part 7:** Difficulty-aware **routing policy** (compute-optimal TTC) quick

## Part 0 — Setup
Set up the environment, load GSM8K and load a lightweight model. For this assignment, we will use Qwen2.5-1.5B Instruct. More information can be found about the model at the link below.

https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct



In [3]:
!pip -q install -U bitsandbytes transformers accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.7 MB/s eta 0:00:00


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset

import torch
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from collections import Counter

import time

# Global timer start
NOTEBOOK_START_TIME = time.time()


### Shared Helper Utilities

To keep this notebook readable and modular, reusable utility functions are stored in:

`shared/helpers.py`

This helper module contains common logic used throughout the notebook, including:

- answer extraction
- example display helpers
- single-generation utilities
- majority voting
- evaluation logic for both the baseline and self-consistency settings

In [29]:
# ## Shared Helper Functions
import sys
from pathlib import Path
import importlib

# Locate repo root by finding shared/helpers.py
_repo_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "helpers.py").exists()),
    None
)

if _repo_root is None:
    raise FileNotFoundError("Could not locate shared/helpers.py from the current notebook directory.")

if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import shared.helpers as helpers
importlib.reload(helpers)

# Pull commonly used names into notebook scope
final_re = helpers.final_re
extract_true_answer = helpers.extract_true_answer
extract_pred_answer = helpers.extract_pred_answer
show_example = helpers.show_example
generate_one = helpers.generate_one
majority_vote = helpers.majority_vote
evaluate = helpers.evaluate
build_generation_pool = helpers.build_generation_pool
evaluate_from_generation_pool = helpers.evaluate_from_generation_pool
evaluate_best_of_n_simple_verifier = helpers.evaluate_best_of_n_simple_verifier
build_difficulty_table_from_pool = helpers.build_difficulty_table_from_pool
build_revision_prompt = helpers.build_revision_prompt
run_revision_chain = helpers.run_revision_chain
select_revision_attempt = helpers.select_revision_attempt
evaluate_revision_chains = helpers.evaluate_revision_chains
build_step_prompt = helpers.build_step_prompt
generate_continuations_with_scores = helpers.generate_continuations_with_scores
beam_search_reasoning = helpers.beam_search_reasoning
evaluate_beam_search = helpers.evaluate_beam_search
run_routed_strategy = helpers.run_routed_strategy
evaluate_routing_policy = helpers.evaluate_routing_policy
summarize_policy_results = helpers.summarize_policy_results

### Load the GSM8K dataset and create train/eval subsets.

In [6]:
# ## Load and Prepare Dataset
dataset = load_dataset("gsm8k", "main")
print("Dataset loaded successfully!")

# Create subsets (adjust sizes if you want)
train_subset = dataset["train"].select(range(1000))
eval_subset  = dataset["test"].select(range(500))

train_df = train_subset.to_pandas()
eval_df  = eval_subset.to_pandas()

print(f"Training subset: {len(train_df)} examples")
print(f"Evaluation subset: {len(eval_df)} examples")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Dataset loaded successfully!
Training subset: 1000 examples
Evaluation subset: 500 examples


## Explore the Data

The GSM8K is a collection of middle school math problems that include questions and answers. It is a common benchmark data set for assessing reasoning models.

First, we can convert the dataset to pandas DataFrames for quick structural inspection.

In [7]:
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])
print("GSM8K train size:", len(train_df))
print("GSM8K test size:", len(test_df))
# Show the first five rows
train_df.head()

GSM8K train size: 7473
GSM8K test size: 1319


,question,answer
0,Natalia sold clips to 48 of her friends in Apr...,Natalia sold 48/2 = <<48/2=24>>24 clips in May...
1,Weng earns $12 an hour for babysitting. Yester...,Weng earns 12/60 = $<<12/60=0.2>>0.2 per minut...
2,Betty is saving money for a new wallet which c...,"In the beginning, Betty has only 100 / 2 = $<<..."
3,"Julie is reading a 120-page book. Yesterday, s...",Maila read 12 x 2 = <<12*2=24>>24 pages today....
4,James writes a 3-page letter to 2 different fr...,He writes each friend 3*2=<<3*2=6>>6 pages a w...


We can inspect a few training examples to confirm the question/answer format.

In [8]:
# Use show_example function from shared/helpers.py to inspect the first three training examples.
for i in range(3):
    show_example(dataset["train"][i])



QUESTION:
 Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

SOLUTION:
 Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72

QUESTION:
 Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?

SOLUTION:
 Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10

QUESTION:
 Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?

SOLUTION:
 In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandparents gave her 15 * 2 = $<<15*2=30>>30.
This means, Betty needs 100 - 50 - 30 

## Answer Pattern and Regex

It is important to understand the structure of a data set so that a reasoning model can be appropriately prompted. GSM8K solutions end with a standardized final-answer line:

- Final numeric answer appears after `####`
- It is on its own line
- It may be an integer or decimal


Because this format is consistent, we can create a regex pattern to reliably extract the true answer with a regular expression. The regex pattern below is defined in shared/helpers.py:

`final_re = re.compile(r"####\s*([-+]?\d+(?:\.\d+)?)")
`


In [9]:
# Regex is defined in shared/helpers.py
print("Final answer regex:", final_re.pattern)


Final answer regex: ####\s*([-+]?\d+(?:\.\d+)?)


Also, notice that each solution not only includes the final answer, but includes reasoning that lead to the final answer. We defined `extract_true_answer` to parse each full solution and keep only the numeric final answer (the value after `####`.)

In [10]:
dataset = dataset.map(lambda x: {"true_answer": extract_true_answer(x)})


Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

Verify that the extracted `true_answer` aligns with the original `answer` text.


In [11]:
for i in range(3):
    show_example(dataset["train"][i])



QUESTION:
 Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

SOLUTION:
 Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72

TRUE ANSWER: 72

QUESTION:
 Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?

SOLUTION:
 Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10

TRUE ANSWER: 10

QUESTION:
 Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?

SOLUTION:
 In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandparents gave her 15 * 2 = $<<15*2=30>>30.
This

## Model Setup

In this section, we load a lightweight instruction-tuned language model:

- **Model:** Qwen2.5-1.5B-Instruct  
- **Type:** Causal Language Model (decoder-only transformer)  
- **Use case:** Step-by-step reasoning and instruction following  

This model is intentionally small (~1.5B parameters) so it can run efficiently in a Colab environment while still demonstrating reasoning capabilities. We will use Quantitization which significantly reduces GPU usage without a severe cost to precision.  We will use **4-bit quantization** via the `bitsandbytes` library.


### Check GPU Availability

In [12]:
# Check GPU Availability
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute Capability:", torch.cuda.get_device_capability(0))


CUDA Available: True
GPU: Tesla T4
Compute Capability: (7, 5)


### Load the model (4-bit quantized)

In [13]:
# Load Model with 4-bit Quantization (Lightweight)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
#model_name = "Qwen/Qwen2.5-0.5B-Instruct"


tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

model.eval()
print("Model loaded:", model_name)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct


### Part 1 — Run Baseline

The baseline is the simplest evaluation setting: the model produces one deterministic response per question, and that single extracted answer is scored directly.

Here we evaluate the model on a subset of the GSM8K evaluation set using:

- `n_samples=1`
- `temperature=0.0`

This gives us a reference point before adding extra inference-time compute through self-consistency.

In [14]:
# ## Run Baseline (System-1)
# Tip: with the 1.5B model, 100 questions should usually be much faster.

baseline_metrics, baseline_results = evaluate(
    eval_df=eval_df,
    model=model,
    tokenizer=tokenizer,
    n_samples=1,
    temperature=0.0,
    max_questions=30
)

# Extract key values
baseline_acc = baseline_metrics["accuracy"]
baseline_q = baseline_metrics["num_questions"]
baseline_correct = int(baseline_acc * baseline_q)

print("Baseline metrics:")
print(json.dumps(baseline_metrics, indent=2))

print("\nSummary:")
print(f"Correct: {baseline_correct} / {baseline_q}")
print(f"Accuracy: {baseline_acc:.2%}")

Eval: majority_vote: 100%|██████████| 30/30 [05:56<00:00, 11.89s/it]

Baseline metrics:
{
  "accuracy": 0.26666666666666666,
  "num_questions": 30,
  "num_correct": 8,
  "n_samples": 1,
  "temperature": 0.0,
  "selection_mode": "majority_vote",
  "elapsed_sec": 356.5745801925659,
  "avg_input_tokens_per_question": 89.5,
  "avg_new_tokens_per_question": 171.8,
  "avg_total_tokens_per_question": 261.3
}

Summary:
Correct: 8 / 30
Accuracy: 26.67%


The baseline model answered **8 out of 30 questions correctly**, achieving an accuracy of **26.67%**.

This provides a simple reference point before introducing additional inference-time compute through multi-sample generation and alternative selection strategies.

## Parts 2 and 3 — Shared Proposal Pool

To compare **majority vote** and **verifier-based selection** more fairly, we first generate a **single frozen pool of candidate solutions** for each question.

In this setup, each question is sampled **5 times** with `temperature=0.7`, producing a shared set of proposals that is reused across both methods:

- **Part 2:** select the final answer using **majority vote**
- **Part 3:** select the final answer using a **simple verifier**

This is a more controlled experimental design than regenerating candidates separately for each section, because any difference in performance is more directly attributable to the **selection rule** rather than to different sampled generations.

In [15]:
# Build one shared proposal pool for Parts 2 and 3
proposal_pool, proposal_pool_meta = build_generation_pool(
    eval_df=eval_df,
    model=model,
    tokenizer=tokenizer,
    n_samples=5,
    temperature=0.7,
    max_questions=30,
    max_new_tokens=256,
)

print("Shared proposal pool built.")
print(json.dumps(proposal_pool_meta, indent=2))

Build generation pool (n=5): 100%|██████████| 30/30 [24:41<00:00, 49.40s/it]

Shared proposal pool built.
{
  "num_questions": 30,
  "n_samples": 5,
  "temperature": 0.7,
  "avg_input_tokens_per_question": 447.5,
  "avg_new_tokens_per_question": 824.4666666666667,
  "avg_total_tokens_per_question": 1271.9666666666667
}


### Part 2 — Self-Consistency from a Shared Proposal Pool

Self-consistency increases inference-time compute by generating multiple candidate solutions for each question and selecting the most common final answer using majority vote.

Here, the candidate solutions are not regenerated inside Part 2. Instead, they come from the shared frozen proposal pool built above. This means the model still uses **5 sampled generations per question**, but those same generations will also be reused in Part 3.

The key idea behind self-consistency is that randomness (`temperature=0.7`) allows the model to explore different reasoning paths. Each generated response is reduced to a final numeric answer, and the most frequently occurring answer is selected as the final prediction.

For example, if the five sampled answers are `[18, 18, 22, 18, 22]`, the final prediction is **18** based on majority vote.

Using a shared pool makes this method easier to compare fairly against verifier-based selection, since both methods operate on the exact same candidate set.

In [16]:
# Part 2: Self-Consistency from the shared proposal pool
sc_metrics, sc_results = evaluate_from_generation_pool(
    generation_pool=proposal_pool,
    pool_meta=proposal_pool_meta,
    selection_mode="majority_vote",
)

sc_acc = sc_metrics["accuracy"]
sc_q = sc_metrics["num_questions"]
sc_correct = sc_metrics["num_correct"]

print("Self-consistency metrics:")
print(json.dumps(sc_metrics, indent=2))

print("\nSummary:")
print(f"Correct: {sc_correct} / {sc_q}")
print(f"Accuracy: {sc_acc:.2%}")

Self-consistency metrics:
{
  "accuracy": 0.43333333333333335,
  "num_questions": 30,
  "num_correct": 13,
  "n_samples": 5,
  "temperature": 0.7,
  "selection_mode": "majority_vote",
  "elapsed_sec": 0.0002262592315673828,
  "avg_input_tokens_per_question": 447.5,
  "avg_new_tokens_per_question": 824.4666666666667,
  "avg_total_tokens_per_question": 1271.9666666666667
}

Summary:
Correct: 13 / 30
Accuracy: 43.33%


### Part 2 Results Summary

Using the shared proposal pool, self-consistency selected the final answer by applying **majority vote** to the same five sampled candidates for each question.

Because the candidate set was frozen in advance, this experiment isolates the effect of the **majority-vote selector** rather than mixing selection effects with newly sampled generations.

## Part 3 — Simple Verifier (Proposer–Verifier Using Labels)

In this section, we compare majority voting with a simple verifier-based selection strategy using the **same frozen proposal pool** from Part 2.

The procedure is:
1. Reuse the same sampled candidate solutions generated earlier
2. Score each candidate with a simple verifier based on the known GSM8K answer
3. Select the candidate with the highest verifier score

This approximates an outcome-based proposer–verifier setup: instead of choosing the most common answer across samples, we choose the single candidate that best matches the label according to the verifier.

### Run Part 3 (Best-of-N + verifier)

In [17]:
# Part 3: Simple verifier from the same shared proposal pool
verifier_metrics, verifier_results = evaluate_from_generation_pool(
    generation_pool=proposal_pool,
    pool_meta=proposal_pool_meta,
    selection_mode="simple_verifier",
    verifier_mode="exact",   # can also try "numeric_distance"
)

verifier_acc = verifier_metrics["accuracy"]
verifier_q = verifier_metrics["num_questions"]
verifier_correct = verifier_metrics["num_correct"]

print("Verifier metrics:")
print(json.dumps(verifier_metrics, indent=2))

print("\nSummary:")
print(f"Correct: {verifier_correct} / {verifier_q}")
print(f"Accuracy: {verifier_acc:.2%}")

Verifier metrics:
{
  "accuracy": 0.6333333333333333,
  "num_questions": 30,
  "num_correct": 19,
  "n_samples": 5,
  "temperature": 0.7,
  "selection_mode": "simple_verifier",
  "elapsed_sec": 0.011571645736694336,
  "avg_input_tokens_per_question": 447.5,
  "avg_new_tokens_per_question": 824.4666666666667,
  "avg_total_tokens_per_question": 1271.9666666666667,
  "verifier_mode": "exact"
}

Summary:
Correct: 19 / 30
Accuracy: 63.33%


## Part 4 — Difficulty Binning (Easy / Medium / Hard)

In this section, we estimate empirical problem difficulty using a small number of stochastic generations per question.

For each question, we:
1. sample the model multiple times with mild randomness,
2. compute the fraction of samples that produce the correct final answer, and
3. assign a difficulty bin based on that empirical pass rate.

We use the following bins:

- **Easy:** pass rate ≥ 0.8  
- **Medium:** 0.3 ≤ pass rate < 0.8  
- **Hard:** pass rate < 0.3

This gives a simple way to approximate which problems are consistently solved by the model and which remain unstable or difficult.

After assigning bins, we compare baseline, self-consistency, and simple-verifier performance across those difficulty groups.

In [20]:
difficulty_df = build_difficulty_table_from_pool(proposal_pool)

print("Difficulty bin counts:")
print(difficulty_df["difficulty"].value_counts())

difficulty_df.head()

Difficulty bin counts:
difficulty
hard      18
easy       6
medium     6
Name: count, dtype: int64


,idx,question,true_answer,pass_rate,difficulty,n_samples_used
0,0,Janet’s ducks lay 16 eggs per day. She eats th...,18,0.0,hard,5
1,1,A robe takes 2 bolts of blue fiber and half th...,3,0.8,easy,5
2,2,Josh decides to try flipping a house. He buys...,70000,0.0,hard,5
3,3,James decides to run 3 sprints 3 times a week....,540,0.8,easy,5
4,4,"Every day, Wendi feeds each of her chickens th...",20,0.4,medium,5


### Difficulty Binning Summary

Difficulty here is defined relative to the model’s own empirical pass rate on each problem.

Because we reuse the same shared proposal pool from Parts 2 and 3, these bins are based on the same sampled candidate sets used in the earlier experiments. This keeps the notebook more consistent and avoids unnecessary extra generation cost.

A high pass rate suggests that the problem is reliably solved by the model, while a low pass rate suggests that the problem remains unstable or difficult even when multiple reasoning paths are sampled.

## Part 5 — Sequential Revisions (Self-Refinement)

In this section, we test whether a model can improve its own answer by revising earlier reasoning.

Instead of sampling several independent solutions at once, we build a **revision chain**:

- **Attempt 0:** baseline solution
- **Attempts 1..R:** revised solutions conditioned on the previous attempt

This gives a different form of test-time compute. Rather than increasing breadth through parallel samples, we increase depth through iterative self-correction.

We evaluate several ways to choose a final answer from the chain:

- **Last revision:** use the final attempt directly
- **Majority vote:** choose the most common predicted answer across all attempts
- **Simple verifier:** choose the attempt with the highest verifier score

In [21]:
# Reuse the existing proposal_pool from Parts 2 and 3
#difficulty_df = build_difficulty_table_from_pool(proposal_pool)

# Attach difficulty labels to the same eval subset used for the pool
eval_with_diff = eval_df.iloc[:len(difficulty_df)].copy().reset_index().rename(columns={"index": "idx"})
eval_with_diff = eval_with_diff.merge(
    difficulty_df[["idx", "difficulty", "pass_rate"]],
    on="idx",
    how="left"
)

eval_with_diff[["idx", "difficulty", "pass_rate"]].head()

,idx,difficulty,pass_rate
0,0,hard,0.0
1,1,easy,0.8
2,2,hard,0.0
3,3,easy,0.8
4,4,medium,0.4


In [22]:
rev_last_metrics, rev_last_rows = evaluate_revision_chains(
    eval_df=eval_with_diff,
    model=model,
    tokenizer=tokenizer,
    r_steps=3,
    base_temperature=0.0,
    rev_temperature=0.7,
    max_questions=5,
    max_new_tokens=256,
    selection_mode="last",
)

print("Revision metrics (last):")
print(json.dumps(rev_last_metrics, indent=2))

Eval revisions: last: 100%|██████████| 5/5 [01:34<00:00, 18.90s/it]

Revision metrics (last):
{
  "method": "sequential_revisions",
  "accuracy": 0.8,
  "num_questions": 5,
  "num_correct": 4,
  "r_steps": 3,
  "base_temperature": 0.0,
  "rev_temperature": 0.7,
  "selection_mode": "last",
  "elapsed_sec": 94.51206064224243,
  "avg_input_tokens_per_question": 1569.2,
  "avg_new_tokens_per_question": 285.0,
  "avg_total_tokens_per_question": 1854.2
}


In [23]:
rev_mv_metrics, rev_mv_rows = evaluate_revision_chains(
    eval_df=eval_with_diff,
    model=model,
    tokenizer=tokenizer,
    r_steps=3,
    base_temperature=0.0,
    rev_temperature=0.7,
    max_questions=5,
    max_new_tokens=256,
    selection_mode="majority_vote",
)

print("Revision metrics (majority vote):")
print(json.dumps(rev_mv_metrics, indent=2))

Eval revisions: majority_vote: 100%|██████████| 5/5 [02:08<00:00, 25.75s/it]

Revision metrics (majority vote):
{
  "method": "sequential_revisions",
  "accuracy": 0.6,
  "num_questions": 5,
  "num_correct": 3,
  "r_steps": 3,
  "base_temperature": 0.0,
  "rev_temperature": 0.7,
  "selection_mode": "majority_vote",
  "elapsed_sec": 128.7657380104065,
  "avg_input_tokens_per_question": 1826.8,
  "avg_new_tokens_per_question": 394.0,
  "avg_total_tokens_per_question": 2220.8
}


In [24]:
rev_ver_metrics, rev_ver_rows = evaluate_revision_chains(
    eval_df=eval_with_diff,
    model=model,
    tokenizer=tokenizer,
    r_steps=3,
    base_temperature=0.0,
    rev_temperature=0.7,
    max_questions=5,
    max_new_tokens=256,
    selection_mode="simple_verifier",
    verifier_mode="exact",
)

print("Revision metrics (simple verifier):")
print(json.dumps(rev_ver_metrics, indent=2))

Eval revisions: simple_verifier: 100%|██████████| 5/5 [01:50<00:00, 22.07s/it]

Revision metrics (simple verifier):
{
  "method": "sequential_revisions",
  "accuracy": 0.8,
  "num_questions": 5,
  "num_correct": 4,
  "r_steps": 3,
  "base_temperature": 0.0,
  "rev_temperature": 0.7,
  "selection_mode": "simple_verifier",
  "elapsed_sec": 110.36621713638306,
  "avg_input_tokens_per_question": 1673.4,
  "avg_new_tokens_per_question": 334.8,
  "avg_total_tokens_per_question": 2008.2,
  "verifier_mode": "exact"
}


### Part 5 Results Summary

Sequential revisions test a different type of inference-time compute from Best-of-N sampling.

- **Best-of-N** explores several independent reasoning paths in parallel
- **Sequential revisions** repeatedly improves one evolving reasoning path

This makes Part 5 useful for comparing **breadth vs. depth** at test time.

If revisions help, that suggests the model can sometimes recognize and repair its own mistakes.
If revisions do not help, that suggests either:
- the initial mistake is persistent across revisions, or
- the model lacks a strong enough self-correction signal.

In the next step, we can compare revision performance against earlier methods under a roughly similar token budget.

## Part 6 — Lightweight Beam/Search

In this section, we explore a simple beam-style search procedure over partial reasoning.

Unlike Best-of-N, which samples several full solutions independently, beam search builds solutions incrementally:

- keep a small beam of partial reasoning states
- expand each state with several candidate next continuations
- rank the expanded states using a simple score based on token log-probability and a small length penalty
- stop when a candidate appears to contain a final numeric answer or when the maximum number of steps is reached

This gives a more structured form of test-time compute than either self-consistency or sequential revisions.

To keep the notebook modular, the Part 6 implementation reuses the existing shared helper stack for:
- answer extraction
- verifier selection
- evaluation bookkeeping


In [28]:
beam_metrics, beam_rows = evaluate_beam_search(
    eval_df=eval_with_diff,
    model=model,
    tokenizer=tokenizer,
    beam_size=3,
    expansions_per_beam=2,
    max_steps=4,
    temperature=0.7,
    step_max_new_tokens=80,
    length_penalty=0.02,
    verifier_mode="exact",
    max_questions=5,   # start small first
)

print("Beam-search metrics:")
print(json.dumps(beam_metrics, indent=2))

Eval beam search (B=3): 100%|██████████| 5/5 [00:57<00:00, 11.44s/it]

Beam-search metrics:
{
  "method": "beam_search",
  "accuracy": 0.0,
  "num_questions": 5,
  "num_correct": 0,
  "beam_size": 3,
  "expansions_per_beam": 2,
  "max_steps": 4,
  "temperature": 0.7,
  "step_max_new_tokens": 80,
  "length_penalty": 0.02,
  "verifier_mode": "exact",
  "elapsed_sec": 57.19004487991333,
  "avg_new_tokens_per_question": 80.0
}


### Part 6 Results Summary

Beam search introduces a more structured search procedure than earlier methods.

- **Best-of-N** increases breadth by sampling several complete solutions
- **Sequential revisions** increases depth by repeatedly refining one trajectory
- **Beam search** keeps several partial reasoning paths alive and expands them step by step

This makes Part 6 useful for comparing a lightweight search procedure against the simpler inference-time strategies from earlier sections.

Because beam search is more expensive, it is best run first on a small subset before scaling up.

## Part 7 — Difficulty-Aware Routing Policy

In this final section, we combine the earlier methods into a simple routing policy.

Instead of applying the same amount of test-time compute to every problem, we use the difficulty bins from Part 4 to decide how much compute to spend:

- **Easy:** use a cheaper strategy
- **Medium:** use a moderate strategy
- **Hard:** use a more expensive strategy

This creates a simple compute-aware policy that can trade off:
- accuracy
- token cost
- runtime

The goal is not only to maximize accuracy, but to do so more efficiently than applying the most expensive strategy uniformly to every question.

In [30]:
policy_A = {"easy": "S1", "medium": "S2", "hard": "S3"}
policy_B = {"easy": "S1", "medium": "S3", "hard": "S4"}
policy_C = {"easy": "S2", "medium": "S3", "hard": "S4"}

cfg_map = {
    "S1": {
        "temperature": 0.0,
        "max_new_tokens": 256,
    },
    "S2": {
        "n_samples": 5,
        "temperature": 0.7,
        "max_new_tokens": 256,
    },
    "S3": {
        "r_steps": 2,
        "base_temperature": 0.0,
        "rev_temperature": 0.7,
        "max_new_tokens": 256,
        "selection_mode": "simple_verifier",
        "verifier_mode": "exact",
    },
    "S4": {
        "beam_size": 3,
        "expansions_per_beam": 2,
        "max_steps": 4,
        "temperature": 0.7,
        "step_max_new_tokens": 80,
        "length_penalty": 0.02,
        "verifier_mode": "exact",
    },
}

In [31]:
# Run on a small subset first

policy_subset = eval_with_diff.copy()

polA_metrics, polA_rows = evaluate_routing_policy(
    eval_df=policy_subset,
    model=model,
    tokenizer=tokenizer,
    policy_map=policy_A,
    cfg_map=cfg_map,
    max_questions=10,
    policy_name="Policy A",
)

polB_metrics, polB_rows = evaluate_routing_policy(
    eval_df=policy_subset,
    model=model,
    tokenizer=tokenizer,
    policy_map=policy_B,
    cfg_map=cfg_map,
    max_questions=10,
    policy_name="Policy B",
)

polC_metrics, polC_rows = evaluate_routing_policy(
    eval_df=policy_subset,
    model=model,
    tokenizer=tokenizer,
    policy_map=policy_C,
    cfg_map=cfg_map,
    max_questions=10,
    policy_name="Policy C",
)

print("Policy A:")
print(json.dumps(polA_metrics, indent=2))

print("\nPolicy B:")
print(json.dumps(polB_metrics, indent=2))

print("\nPolicy C:")
print(json.dumps(polC_metrics, indent=2))

Eval routing: Policy C: 100%|██████████| 10/10 [03:07<00:00, 18.79s/it]

Policy A:
{
  "method": "difficulty_aware_routing",
  "policy_name": "Policy A",
  "accuracy": 0.4,
  "num_questions": 10,
  "num_correct": 4,
  "avg_new_tokens_per_question": 327.2,
  "elapsed_sec": 201.51864171028137,
  "policy_map": {
    "easy": "S1",
    "medium": "S2",
    "hard": "S3"
  }
}

Policy B:
{
  "method": "difficulty_aware_routing",
  "policy_name": "Policy B",
  "accuracy": 0.4,
  "num_questions": 10,
  "num_correct": 4,
  "avg_new_tokens_per_question": 110.8,
  "elapsed_sec": 103.76355910301208,
  "policy_map": {
    "easy": "S1",
    "medium": "S3",
    "hard": "S4"
  }
}

Policy C:
{
  "method": "difficulty_aware_routing",
  "policy_name": "Policy C",
  "accuracy": 0.5,
  "num_questions": 10,
  "num_correct": 5,
  "avg_new_tokens_per_question": 251.2,
  "elapsed_sec": 187.89367032051086,
  "policy_map": {
    "easy": "S2",
    "medium": "S3",
    "hard": "S4"
  }
}


In [32]:
# Add a compact comparison table cell
policy_summary_df = summarize_policy_results({
    "Policy A": polA_metrics,
    "Policy B": polB_metrics,
    "Policy C": polC_metrics,
})

policy_summary_df

,policy,accuracy,num_correct,num_questions,avg_new_tokens_per_question,elapsed_sec
2,Policy C,0.5,5,10,251.2,187.893670
1,Policy B,0.4,4,10,110.8,103.763559
0,Policy A,0.4,4,10,327.2,201.518642


### Part 7 Results Summary

Difficulty-aware routing lets us allocate compute more selectively across problems.

Compared with a uniform strategy, a routing policy can sometimes achieve a better tradeoff between:
- overall accuracy
- average token usage
- runtime

This makes Part 7 the most practical section of the notebook, because it turns the earlier TTC methods into a simple decision policy rather than treating every problem the same way.

#
### Save results

In [ ]:
# ## Save Results

final_output = {
    "model_name": model_name,
    "baseline_metrics": baseline_metrics,
    "baseline_results": baseline_results,
    "self_consistency_metrics": sc_metrics,
    "self_consistency_results": sc_results,
}

with open("gsm8k_refactored_results.json", "w") as f:
    json.dump(final_output, f, indent=2)

print("Saved: gsm8k_refactored_results.json")


## Results Summary + Plots

Fill this section with your final runs (200–500 eval questions).  
Minimum plots/tables to include in your report:

- Accuracy vs. Avg tokens for:
  - Baseline (Part 1)
  - Best-of-N majority (Part 2)
  - Best-of-N verifier (Part 3)
  - Revisions (Part 5)
  - Beam/search (Part 6; on a subset is fine)
- Accuracy by difficulty bin (Easy/Medium/Hard)
- Policy routing comparison (Part 7)


In [ ]:
# Collect the metrics you ran above into one DataFrame for quick comparison.
metric_rows = []

# These variables exist if you ran the example cells; add/remove as needed.
for m in [baseline_metrics, sc_metrics, bover_metrics, rev_metrics, beam_metrics]:
    if isinstance(m, dict):
        metric_rows.append(m)

metrics_df = pd.DataFrame(metric_rows)
metrics_df


In [ ]:
# Simple plot: accuracy vs avg_tokens
if "accuracy" in metrics_df.columns and "avg_tokens" in metrics_df.columns:
    plt.figure()
    plt.scatter(metrics_df["avg_tokens"], metrics_df["accuracy"])
    for _, r in metrics_df.iterrows():
        label = r.get("method","")
        plt.annotate(label, (r["avg_tokens"], r["accuracy"]))
    plt.xlabel("Avg tokens (TTC proxy)")
    plt.ylabel("Accuracy")
    plt.title("Accuracy vs Test-Time Compute (tokens)")
    plt.show()


## Report Template (3–5 pages)

Use this structure (you can paste into a Google Doc or write directly here):

1. **Motivation**
   - What is test-time compute (TTC)?
   - Why GSM8K and why a lightweight open model?
   - What you expect: sampling, selection, revisions, and search trade compute for accuracy.

2. **Methods**
   - Dataset split sizes and preprocessing
   - Model details (Qwen2.5-1.5B, quantized), decoding settings
   - Implemented strategies (Parts 1–7), with key hyperparameters (N, R, B, M, etc.)
   - Compute metric: average generated tokens (and/or wall-clock, if recorded)

3. **Results**
   - Overall accuracy vs compute tradeoff for each method
   - Breakdown by difficulty bin (Easy/Medium/Hard)
   - Policy routing comparison vs uniform strategies

4. **Discussion**
   - Where did TTC help most (which difficulty bins)?
   - Majority vote vs verifier selection: when does each win?
   - Revisions vs Best-of-N under similar compute budgets
   - Is beam/search “worth it” for this model/budget?
   - Connection back to the behaviors described in Snell-style TTC work

5. **Limitations**
   - Token proxy vs true FLOPs/time
   - Heuristic scoring for beam search (no learned PRM)
   - Small model constraints and prompt sensitivity


In [ ]:
# Optional: export your metrics and selected rows to JSON for your report
out = {
    "metrics_table": metrics_df.to_dict(orient="records"),
    "policy_A": polA_metrics,
    "policy_B": polB_metrics,
    "policy_C": polC_metrics,
}
with open("hw_results_summary.json","w") as f:
    json.dump(out, f, indent=2)
print("Wrote hw_results_summary.json")


In [ ]:
NOTEBOOK_END_TIME = time.time()
total_runtime = NOTEBOOK_END_TIME - NOTEBOOK_START_TIME

print("\n===== TOTAL NOTEBOOK RUNTIME =====")
print("Total time (seconds):", round(total_runtime, 2))
print("Total time (minutes):", round(total_runtime / 60, 2))